# Principio de Inversión de Dependencias (DIP)

Los módulos de alto nivel deben depender de abstracciones, no de implementaciones concretas.

## ❌ Sin aplicar DIP

El servicio crea directamente un notificador de correo. Cambiar de canal obliga a modificar el módulo de alto nivel.

In [ ]:
class NotificadorCorreo:
    def __init__(self, remitente, servidor): self.remitente = remitente; self.servidor = servidor
    def construir(self, destinatario, mensaje): return f'De {self.remitente} para {destinatario}: {mensaje}'
    def enviar(self, destinatario, mensaje): return f'[Correo/{self.servidor}] {self.construir(destinatario, mensaje)}'

class ServicioPedidoAcoplado:
    def __init__(self, tienda, impuesto):
        self.tienda = tienda; self.impuesto = impuesto; self.notificador = NotificadorCorreo('pedidos@tienda.co','smtp.tienda.co')
    def calcular_total(self, subtotal): return subtotal * (1 + self.impuesto)
    def confirmar(self, cliente, subtotal):
        return self.notificador.enviar(cliente, f'Pedido confirmado por ${self.calcular_total(subtotal):,.0f}')

servicio_mal = ServicioPedidoAcoplado('Comida Express',0.08)
print(servicio_mal.confirmar('ana@email.com',50000))
print('Para usar SMS habría que editar ServicioPedidoAcoplado.')

## ✅ Aplicando DIP

El servicio recibe el contrato `Notificador` por el constructor. Correo y SMS son detalles intercambiables.

In [ ]:
from abc import ABC, abstractmethod

class Notificador(ABC):
    def __init__(self, origen, activo=True): self.origen = origen; self.activo = activo
    @abstractmethod
    def construir(self, destinatario, mensaje): pass
    @abstractmethod
    def enviar(self, destinatario, mensaje): pass

class NotificadorCorreoDIP(Notificador):
    def __init__(self, origen, servidor): super().__init__(origen); self.servidor = servidor
    def construir(self, destinatario, mensaje): return f'De {self.origen} para {destinatario}: {mensaje}'
    def enviar(self, destinatario, mensaje): return f'[Correo/{self.servidor}] {self.construir(destinatario,mensaje)}'

class NotificadorSMS(Notificador):
    def __init__(self, origen, proveedor): super().__init__(origen); self.proveedor = proveedor
    def construir(self, destinatario, mensaje): return f'{self.origen} -> {destinatario}: {mensaje}'
    def enviar(self, destinatario, mensaje): return f'[SMS/{self.proveedor}] {self.construir(destinatario,mensaje)}'

class ServicioPedido:
    def __init__(self, tienda, notificador, impuesto=0.0): self.tienda = tienda; self.notificador = notificador; self.impuesto = impuesto
    def calcular_total(self, subtotal): return subtotal * (1 + self.impuesto)
    def confirmar(self, cliente, subtotal):
        mensaje = f'{self.tienda}: pedido confirmado por ${self.calcular_total(subtotal):,.0f}'
        return self.notificador.enviar(cliente, mensaje)

correo = NotificadorCorreoDIP('pedidos@tienda.co','smtp.tienda.co')
sms = NotificadorSMS('Comida Express','MensajesCo')
print(ServicioPedido('Comida Express',correo,0.08).confirmar('ana@email.com',50000))
print(ServicioPedido('Comida Express',sms,0.08).confirmar('+57 300 000 0000',50000))

La clase de alto nivel no cambia al sustituir correo por SMS: depende de la abstracción y recibe el detalle desde afuera.